# LeetCode #47: Permutations II

https://leetcode.com/problems/permutations-ii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n \times n!)$ | $O(n \times n!)$ |
| **Optimal: Backtracking + Deduplication ★** | $O(n \times n!)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Generate every permutation (including duplicates), insert each into a hash set, then convert the set to a list. Works but wastes time building duplicate permutations only to throw them away, and requires $O(n \times n!)$ extra memory for the set.

### Optimal: Backtracking + Deduplication ★
Sort the array so duplicate values are adjacent, then use a boolean `used[]` array. At each recursion level skip element `i` if `nums[i] == nums[i-1]` AND `used[i-1]` is false — this means the previous identical element was already "un-chosen" at this level, so choosing `nums[i]` now would generate a duplicate subtree.

**Why this is better than Brute Force:** Pruning cuts entire duplicate subtrees before they are explored, so no post-processing set is needed and memory stays $O(n)$ for the call stack and path.

**Constraints:**
* 1 <= nums.length <= 8
* -10 <= nums[i] <= 10

## Solutions

### C#

In [ ]:
public class Solution {
    public IList<IList<int>> PermuteUnique(int[] nums) {
        Array.Sort(nums);
        var result = new List<IList<int>>();
        var used = new bool[nums.Length];
        Backtrack(nums, used, new List<int>(), result);
        return result;
    }

    private void Backtrack(int[] nums, bool[] used, List<int> path, IList<IList<int>> result) {
        if (path.Count == nums.Length) {
            result.Add(new List<int>(path));
            return;
        }
        for (int i = 0; i < nums.Length; i++) {
            if (used[i]) continue;
            // Skip duplicate: same value as previous element, and previous was NOT used at this level
            // This ensures only the first copy of each duplicate value starts a new branch here
            if (i > 0 && nums[i] == nums[i - 1] && !used[i - 1]) continue;
            used[i] = true;
            path.Add(nums[i]);
            Backtrack(nums, used, path, result);
            // Undo this choice before trying the next element
            path.RemoveAt(path.Count - 1);
            used[i] = false;
        }
    }
}

### Python

In [ ]:
class Solution:
    def permuteUnique(self, nums: list[int]) -> list[list[int]]:
        nums.sort()
        result = []
        used = [False] * len(nums)

        def backtrack(path: list[int]) -> None:
            if len(path) == len(nums):
                result.append(list(path))
                return
            for i in range(len(nums)):
                if used[i]:
                    continue
                # Skip duplicate: same value as previous element, and previous was NOT used at this level
                # This ensures only the first copy of each duplicate value starts a new branch here
                if i > 0 and nums[i] == nums[i - 1] and not used[i - 1]:
                    continue
                used[i] = True
                path.append(nums[i])
                backtrack(path)
                # Undo this choice before trying the next element
                path.pop()
                used[i] = False

        backtrack([])
        return result

### Go

In [ ]:
func permuteUnique(nums []int) [][]int {
    sort.Ints(nums)
    result := [][]int{}
    used := make([]bool, len(nums))
    path := []int{}

    var backtrack func()
    backtrack = func() {
        if len(path) == len(nums) {
            perm := make([]int, len(path))
            copy(perm, path)
            result = append(result, perm)
            return
        }
        for i := 0; i < len(nums); i++ {
            if used[i] {
                continue
            }
            // Skip duplicate: same value as previous element, and previous was NOT used at this level
            // This ensures only the first copy of each duplicate value starts a new branch here
            if i > 0 && nums[i] == nums[i-1] && !used[i-1] {
                continue
            }
            used[i] = true
            path = append(path, nums[i])
            backtrack()
            // Undo this choice before trying the next element
            path = path[:len(path)-1]
            used[i] = false
        }
    }
    backtrack()
    return result
}

### Rust

In [ ]:
impl Solution {
    pub fn permute_unique(mut nums: Vec<i32>) -> Vec<Vec<i32>> {
        nums.sort();
        let n = nums.len();
        let mut result = Vec::new();
        let mut used = vec![false; n];
        let mut path = Vec::new();
        Self::backtrack(&nums, &mut used, &mut path, &mut result);
        result
    }

    fn backtrack(nums: &[i32], used: &mut Vec<bool>, path: &mut Vec<i32>, result: &mut Vec<Vec<i32>>) {
        if path.len() == nums.len() {
            result.push(path.clone());
            return;
        }
        for i in 0..nums.len() {
            if used[i] { continue; }
            // Skip duplicate: same value as previous element, and previous was NOT used at this level
            // This ensures only the first copy of each duplicate value starts a new branch here
            if i > 0 && nums[i] == nums[i - 1] && !used[i - 1] { continue; }
            used[i] = true;
            path.push(nums[i]);
            Self::backtrack(nums, used, path, result);
            // Undo this choice before trying the next element
            path.pop();
            used[i] = false;
        }
    }
}

## Example Scenarios

**1. Common Case**
**Input:** nums = [1, 1, 2]
After sorting: [1, 1, 2]. At depth 0, index 0 (value 1) is chosen; index 1 is skipped because nums[1]==nums[0] and used[0] is false — this prunes the duplicate branch. Only two distinct first choices exist (1 and 2), yielding 3 unique permutations: [1,1,2], [1,2,1], [2,1,1].

**2. Slightly Complex**
**Input:** nums = [1, 2, 3] (no duplicates)
The deduplication guard never fires because no two adjacent sorted values are equal. Behavior is identical to #46 (Permutations), generating all $3! = 6$ permutations.

**3. Edge Case: Time Factor**
**Input:** nums = [1, 1, 1, 1, 1, 1, 1, 1] (eight identical elements, maximum n = 8)
Every permutation is [1,1,1,1,1,1,1,1], so only 1 unique result exists. The deduplication guard fires at every level after the first choice, cutting the tree from $8! = 40320$ leaves to exactly 1. This is the most aggressive pruning possible.

**4. Edge Case: Space Factor**
**Input:** nums = [1, 2, 3, 4, 5, 6, 7, 8] (all distinct, maximum n = 8)
$8! = 40320$ unique permutations are produced. Stack depth is 8; the `used` and `path` arrays each hold 8 elements. Output size dominates: $40320 \times 8 = 322560$ integers stored. The extra memory for deduplication is $O(n)$, much less than the $O(n \times n!)$ a set-based approach would need.

**5. Almost-Impossible but Plausible**
**Input:** nums = [-10, -10, 0, 0, 10, 10]
Six elements with three pairs of duplicates. The $6! / (2! \times 2! \times 2!) = 90$ unique permutations are produced. The sort brings pairs together, and the deduplication guard fires reliably for each pair, preventing $6! - 90 = 630$ redundant subtrees from being explored.